In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

# Layer experiments

Compare the TinyTorch implementations of `Linear_CP`, `Dropout_CP`, and `Sequential` with their PyTorch equivalents. Each section checks correctness first and then measures CPU forward-pass time.

In [3]:
from statistics import median
from time import perf_counter

import numpy as np
import torch
from torch import nn

from tinytorch.activations import ReLU_CP
from tinytorch.layers import Dropout_CP, Linear_CP, Sequential
from tinytorch.tensor import Tensor_CP

np.random.seed(42)
torch.manual_seed(42)

print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print("Device:  CPU")

NumPy:  2.5.0
PyTorch: 2.12.1+cpu
Device:  CPU


### Timing helper

The timer warms up each operation and reports the median time per call. Inputs and layers are created outside the timed region, so the measurements focus on `forward()`.

In [4]:
def benchmark(function, *, warmup=2, repeats=5, number=10):
    for _ in range(warmup):
        function()

    samples_ms = []
    for _ in range(repeats):
        start = perf_counter()
        for _ in range(number):
            function()
        samples_ms.append((perf_counter() - start) * 1_000 / number)

    return median(samples_ms)


def print_benchmark(name, tiny_ms, torch_ms):
    ratio = tiny_ms / torch_ms
    print(f"{name:<12} TinyTorch: {tiny_ms:>9.4f} ms")
    print(f"{'':<12} PyTorch:   {torch_ms:>9.4f} ms")
    print(f"{'':<12} ratio:     {ratio:>9.2f}x (TinyTorch / PyTorch)")

### Testing Linear

TinyTorch stores a linear weight as `(in_features, out_features)`, while PyTorch stores it as `(out_features, in_features)`. The weight is transposed when copied so both layers perform the same calculation.

In [5]:
rng = np.random.default_rng(7)
x_np = rng.normal(size=(4, 5)).astype(np.float32)
weight_np = rng.normal(size=(5, 3)).astype(np.float32)
bias_np = rng.normal(size=(3,)).astype(np.float32)

linear_cp = Linear_CP(5, 3)
linear_cp.weight = Tensor_CP(weight_np)
linear_cp.bias = Tensor_CP(bias_np)

linear_torch = nn.Linear(5, 3)
linear_torch.requires_grad_(False)
linear_torch.weight.copy_(torch.from_numpy(weight_np.T))
linear_torch.bias.copy_(torch.from_numpy(bias_np))

y_cp = linear_cp(Tensor_CP(x_np))
y_torch = linear_torch(torch.from_numpy(x_np))
max_difference = np.max(np.abs(y_cp.data - y_torch.numpy()))

print("PyTorch Linear:\n", y_torch.numpy())
print("\nTinyTorch Linear:\n", y_cp.data)
print(f"\nMax absolute difference: {max_difference:.2e}")

assert y_cp.shape == tuple(y_torch.shape)
assert np.allclose(y_cp.data, y_torch.numpy(), atol=1e-5), "Linear output mismatch!"
assert len(linear_cp.parameters()) == len(list(linear_torch.parameters())) == 2
print("Linear output and parameter count match PyTorch")

PyTorch Linear:
 [[ 0.30516982  1.892175    0.7834054 ]
 [-1.7864941   0.7430747   1.6418967 ]
 [-1.9549925   1.2988796   0.60521626]
 [-0.25461578  3.791981   -0.18315315]]

TinyTorch Linear:
 [[ 0.3051697   1.892175    0.7834055 ]
 [-1.7864941   0.7430747   1.6418967 ]
 [-1.9549925   1.2988796   0.60521626]
 [-0.25461578  3.791981   -0.18315327]]

Max absolute difference: 1.19e-07
Linear output and parameter count match PyTorch


### Linear efficiency

In [6]:
batch_size, in_features, out_features = 32, 64, 32
x_bench_np = rng.normal(size=(batch_size, in_features)).astype(np.float32)
x_bench_cp = Tensor_CP(x_bench_np)
x_bench_torch = torch.from_numpy(x_bench_np)

linear_bench_cp = Linear_CP(in_features, out_features)
linear_bench_torch = nn.Linear(in_features, out_features).requires_grad_(False)
linear_bench_torch.weight.copy_(torch.from_numpy(linear_bench_cp.weight.data.T))
linear_bench_torch.bias.copy_(torch.from_numpy(linear_bench_cp.bias.data))

tiny_ms = benchmark(lambda: linear_bench_cp(x_bench_cp), number=3)
torch_ms = benchmark(lambda: linear_bench_torch(x_bench_torch), number=100)
print_benchmark("Linear", tiny_ms, torch_ms)

Linear       TinyTorch:   28.5889 ms
             PyTorch:      0.0064 ms
             ratio:       4452.41x (TinyTorch / PyTorch)


### Testing Dropout

Exact element-by-element equality is not expected because NumPy and PyTorch use different random-number generators. Instead, we check the important statistical behavior: approximately `p` of the values are zero and inverted dropout keeps the output mean close to the input mean.

In [7]:
p = 0.25
sample_count = 200_000
ones_np = np.ones(sample_count, dtype=np.float32)

np.random.seed(123)
torch.manual_seed(123)
dropout_cp = Dropout_CP(p=p)
dropout_torch = nn.Dropout(p=p).train()

y_cp = dropout_cp(Tensor_CP(ones_np), training=True).data
y_torch = dropout_torch(torch.from_numpy(ones_np)).numpy()

cp_zero_fraction = np.mean(y_cp == 0)
torch_zero_fraction = np.mean(y_torch == 0)

print(f"Expected zero fraction: {p:.3f}")
print(f"TinyTorch zero fraction: {cp_zero_fraction:.3f}, mean: {y_cp.mean():.3f}")
print(f"PyTorch zero fraction:   {torch_zero_fraction:.3f}, mean: {y_torch.mean():.3f}")
print(f"Kept-value scale:         {1 / (1 - p):.3f}")

assert abs(cp_zero_fraction - p) < 0.01
assert abs(torch_zero_fraction - p) < 0.01
assert abs(y_cp.mean() - 1.0) < 0.02
assert abs(y_torch.mean() - 1.0) < 0.02
assert np.allclose(dropout_cp(Tensor_CP(ones_np), training=False).data, ones_np)
print("Dropout statistics and inference behavior match PyTorch")

Expected zero fraction: 0.250
TinyTorch zero fraction: 0.251, mean: 0.999
PyTorch zero fraction:   0.250, mean: 1.000
Kept-value scale:         1.333
Dropout statistics and inference behavior match PyTorch


### Dropout efficiency

In [8]:
dropout_input_np = rng.normal(size=(256, 256)).astype(np.float32)
dropout_input_cp = Tensor_CP(dropout_input_np)
dropout_input_torch = torch.from_numpy(dropout_input_np)
dropout_bench_cp = Dropout_CP(p=0.5)
dropout_bench_torch = nn.Dropout(p=0.5).train()

tiny_ms = benchmark(lambda: dropout_bench_cp(dropout_input_cp, training=True), number=20)
torch_ms = benchmark(lambda: dropout_bench_torch(dropout_input_torch), number=20)
print_benchmark("Dropout", tiny_ms, torch_ms)

Dropout      TinyTorch:    0.4545 ms
             PyTorch:      0.6571 ms
             ratio:          0.69x (TinyTorch / PyTorch)


### Testing Sequential

Both models use the architecture `Linear(4, 5) -> ReLU -> Linear(5, 2)`. Their linear parameters are synchronized before comparing the outputs.

In [9]:
def copy_linear_to_torch(source, target):
    target.weight.copy_(torch.from_numpy(source.weight.data.T))
    if source.bias is not None:
        target.bias.copy_(torch.from_numpy(source.bias.data))


sequential_cp = Sequential(
    Linear_CP(4, 5),
    ReLU_CP(),
    Linear_CP(5, 2),
)
sequential_torch = nn.Sequential(
    nn.Linear(4, 5),
    nn.ReLU(),
    nn.Linear(5, 2),
).requires_grad_(False)

copy_linear_to_torch(sequential_cp.layers[0], sequential_torch[0])
copy_linear_to_torch(sequential_cp.layers[2], sequential_torch[2])

x_np = rng.normal(size=(3, 4)).astype(np.float32)
y_cp = sequential_cp.forward(Tensor_CP(x_np))
y_torch = sequential_torch(torch.from_numpy(x_np))

cp_parameter_count = sum(parameter.size for parameter in sequential_cp.parameters())
torch_parameter_count = sum(parameter.numel() for parameter in sequential_torch.parameters())

print("PyTorch Sequential:\n", y_torch.numpy())
print("\nTinyTorch Sequential:\n", y_cp.data)
print(f"\nParameters: TinyTorch={cp_parameter_count}, PyTorch={torch_parameter_count}")

assert np.allclose(y_cp.data, y_torch.numpy(), atol=1e-5), "Sequential output mismatch!"
assert cp_parameter_count == torch_parameter_count
assert len(sequential_cp.parameters()) == len(list(sequential_torch.parameters())) == 4
print("Sequential output and collected parameters match PyTorch")

PyTorch Sequential:
 [[ 0.13163981 -0.21305211]
 [ 0.28389636 -0.13747314]
 [ 0.02277317  0.13407952]]

TinyTorch Sequential:
 [[ 0.13163982 -0.21305214]
 [ 0.28389636 -0.13747314]
 [ 0.02277317  0.1340795 ]]

Parameters: TinyTorch=37, PyTorch=37
Sequential output and collected parameters match PyTorch


### Sequential efficiency

In [10]:
sequential_bench_cp = Sequential(
    Linear_CP(32, 48),
    ReLU_CP(),
    Linear_CP(48, 16),
)
sequential_bench_torch = nn.Sequential(
    nn.Linear(32, 48),
    nn.ReLU(),
    nn.Linear(48, 16),
).requires_grad_(False)
copy_linear_to_torch(sequential_bench_cp.layers[0], sequential_bench_torch[0])
copy_linear_to_torch(sequential_bench_cp.layers[2], sequential_bench_torch[2])

sequential_input_np = rng.normal(size=(16, 32)).astype(np.float32)
sequential_input_cp = Tensor_CP(sequential_input_np)
sequential_input_torch = torch.from_numpy(sequential_input_np)

tiny_ms = benchmark(lambda: sequential_bench_cp.forward(sequential_input_cp), number=3)
torch_ms = benchmark(lambda: sequential_bench_torch(sequential_input_torch), number=100)
print_benchmark("Sequential", tiny_ms, torch_ms)

Sequential   TinyTorch:   16.3103 ms
             PyTorch:      0.0171 ms
             ratio:        951.65x (TinyTorch / PyTorch)


### Results

- `Linear_CP` matches PyTorch after accounting for the transposed weight layout.
- `Dropout_CP` has the expected drop rate, inverted scaling, and inference behavior.
- `Sequential` produces the same output and collects the same number of parameters.
- PyTorch is expected to be much faster for linear layers because it dispatches matrix multiplication to optimized compiled kernels, while `Tensor_CP.matmul` currently uses three Python loops.

Timings depend on the CPU, thread settings, and system load, so rerun the benchmark cells on the target machine rather than treating one run as a universal result.